In [2]:
import pandas as pd
import numpy as np
import os

current_dir = os.getcwd()
folder_path = os.path.join(current_dir, '..', 'data_raw')

# ĐỌC VÀ GỘP DỮ LIỆU (MERGE)
if not os.path.exists(folder_path):
    print(f"Lỗi: Không tìm thấy thư mục {folder_path}")
else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    file_list.sort()
    
    all_df = []
    for file_name in file_list:
        full_path = os.path.join(folder_path, file_name)
        temp_df = pd.read_csv(full_path)
        # Chuẩn hóa tên cột: viết thường, bỏ khoảng trắng, thay bằng dấu gạch dưới
        temp_df.columns = [c.lower().strip().replace(' ', '_') for c in temp_df.columns]
        all_df.append(temp_df)

    df = pd.concat(all_df, ignore_index=True)
    # Loại bỏ dòng trùng lặp nếu có
    df = df.drop_duplicates()
    print(f"--- Đã gộp xong {len(file_list)} file. Tổng số dòng: {len(df)} ---")

    # FEATURE ENGINEERING (TÁCH CỘT & TẠO BIẾN MỚI)
    df['local_time'] = pd.to_datetime(df['local_time'])
    
    df['date'] = df['local_time'].dt.date
    df['year'] = df['local_time'].dt.year
    df['month'] = df['local_time'].dt.month
    df['hour'] = df['local_time'].dt.hour
    df['day_of_week'] = df['local_time'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

    # Thêm cột Mùa (Season)
    def get_season(m):
        if m in [12, 1, 2]: return 0 # Đông
        elif m in [3, 4, 5]: return 1 # Xuân
        elif m in [6, 7, 8]: return 2 # Hạ
        else: return 3 # Thu
    df['season'] = df['month'].apply(get_season)

    # Thêm cột Phân loại AQI (Cho bài toán Classification)
    def get_aqi_label(v):
        if v <= 50: return 'Good'
        elif v <= 100: return 'Moderate'
        elif v <= 150: return 'Unhealthy for sensitive groups'
        elif v <= 200: return 'Unhealthy'
        elif v <= 300: return 'Very Unhealthy'
        else: return 'Hazardous'
    if 'aqi' in df.columns:
        df['aqi_category'] = df['aqi'].apply(get_aqi_label)

    # Xử lý Missing Values 
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    # Nội suy giữa các dòng
    df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')
    # Vét nốt các ô NaN ở cực đầu hoặc cực cuối file
    df[numeric_cols] = df[numeric_cols].ffill().bfill()

    # Xử lý Outliers 
    pollutants_limits = {
        'co': 2000.0, 'no2': 500.0, 'o3': 400.0, 
        'pm10': 600.0, 'pm25': 500.0, 'so2': 300.0
    }
    for col, limit in pollutants_limits.items():
        if col in df.columns:
            df[col] = df[col].clip(lower=0, upper=limit)

    # Bỏ cột utc_time
    if 'utc_time' in df.columns:
        df = df.drop(columns=['utc_time'])

    # Đưa các cột định danh và thời gian lên đầu
    time_feat = ['local_time', 'date', 'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'season']
    other_cols = [c for c in df.columns if c not in time_feat]
    df = df[time_feat + other_cols]

    # Lưu file sạch
    df.to_csv('hanoi_aqi_cleaned.csv', index=False, encoding='utf-8-sig')

    print("\n--- HOÀN THÀNH CLEAN DATA ---")
    print(f"Số lượng giá trị thiếu còn lại: {df.isnull().sum().sum()}")
    print(f"Số lượng cột hiện có: {len(df.columns)}")
    df.head()

--- Đã gộp xong 4 file. Tổng số dòng: 30341 ---

--- HOÀN THÀNH CLEAN DATA ---
Số lượng giá trị thiếu còn lại: 0
Số lượng cột hiện có: 26
